# The Privileged Access Hypothesis

**Survey Reference:** Section 6.3 - The Privileged Access Hypothesis

A model used to interpret itself can, under appropriate elicitation, access internal variables and intermediate computations that are not reliably recoverable from external behavioral analysis alone, providing a comparative advantage as an interpreter of its own behavior.


## Feature Descriptions Task

This notebook demonstrates the Privileged Access Hypothesis through the **Feature Descriptions Task** from [Li et al., 2025](https://arxiv.org/abs/2511.08579). We compare three different explainer models attempting to describe the internal features of a target model (Llama-3.1-8B):

### Explainer Models Compared:
| Explainer Model | Target Model | Relationship | Hypothesis |
|-----------------|--------------|--------------|------------|
| **Llama-3.1-8B** | Llama-3.1-8B | **Self-explanation** | Should have privileged access |
| **Llama-3.1-8B-Instruct** | Llama-3.1-8B | Instruction-Tuned | Limited access |
| **Llama-3-8B** | Llama-3.1-8B | Different Model | Least access |

## Experimental Tasks

We test the privileged access hypothesis on two complementary tasks:

### 1. **Residual Stream Feature Description** 
- Extract hidden states from layer 15 of the target model
- Ask explainer models to describe what these hidden states encode
- Compare quality of descriptions across self vs. external explainers

### 2. **Sparse Autoencoder (SAE) Feature Description**
- Use pre-trained SAE features from the target model 
- Compare explainer descriptions to ground truth Neuronpedia labels

If the Privileged Access Hypothesis holds, we expect:
- **Self-explainer** can produce the most accurate and detailed descriptions
- **External explainers** show degraded performance, with more distant models performing worse


## 1. Experimental Setup

### Model Configuration

We'll load three different explainer models and configure them to describe features from the target model.

In [1]:
from model.utils import load_models
from transformers import AutoTokenizer
import yaml

config_path = "model/base_131k.yaml"
target_model_path = "meta-llama/Llama-3.1-8B"
target_model_name = "Llama-3.1-8B"

feature_explainer_models = {
    "Llama-3.1-8B": "Transluce/features_explain_llama3.1_8b_llama3.1_8b",
    "Llama-3.1-8B-Instruction": "Transluce/features_explain_llama3.1_8b_llama3.1_8b_instruct",
    "Llama-3-8B": "Transluce/features_explain_llama3.1_8b_llama3_8b",
}

with open(config_path, "r") as f:
    config = yaml.safe_load(f)


### Prompt Templates

The explainer models use special tokens to denote continuous features that need to be described. We define templates that will be used to prompt the models for feature descriptions.

In [2]:
# Define the layer to extract features from and set up prompt templates
# The templates use special tokens to denote continuous features that need to be described
layer = 15
special_tokens = config["continuous_tokens"]

prompt_args = {
    "layer": str(layer),
    "begin_continuous": special_tokens["begin_continuous"],
    "end_continuous": special_tokens["end_continuous"],
    "feature": special_tokens["continuous_rep"],
}

# Three different prompt templates for variety in generation
templates = [
    "We can describe {begin_continuous}{feature}{end_continuous} at layer {layer} as encoding ",
    "Generate a description of this feature at layer {layer}: {begin_continuous}{feature}{end_continuous}.",
    "What does {begin_continuous}{feature}{end_continuous} encode at layer {layer}?",
]
input_with_placeholder = [template.format(**prompt_args) for template in templates]

## 2. Task 1: Residual Stream Feature Description

### Feature Extraction Function

We first define a function to extract hidden states (residual stream activations) from a specific layer of the target model. These hidden states represent the model's internal computations at that layer.

In [3]:
from numpy import extract
import torch
from transformers import AutoModelForCausalLM

@torch.inference_mode()
def extract_from_residual_stream(model_path, layer, texts):
    """
    Extract hidden states from a specific layer of the residual stream.
    
    Args:
        model_path: HuggingFace model identifier
        layer: Which transformer layer to extract from (0-indexed)
        texts: List of input texts to process
    
    Returns:
        Tensor of hidden states [batch_size, hidden_dim] from the last token
    """
    # Load target model and tokenizer
    target_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype="auto")
    target_model_tokenizer = AutoTokenizer.from_pretrained(model_path)
    target_model_tokenizer.pad_token = target_model_tokenizer.eos_token
    target_model_tokenizer.padding_side = "left"
    
    # Tokenize inputs
    inputs = target_model_tokenizer(texts, return_tensors="pt", padding=True).to(target_model.device)
    target_model.eval()
    
    # Storage for hidden states
    hidden_states = []
    
    def hook_fn(module, input, output):
        """Hook function to capture hidden states from the specified layer"""
        # Extract hidden states from the last token position
        if isinstance(output, tuple):
            hidden_states.append(output[0][:, -1].detach().cpu())
        else:
            hidden_states.append(output[:, -1].detach().cpu())
    
    # Register hook on the target layer
    hook_handle = target_model.model.layers[layer].register_forward_hook(hook_fn)
    
    # Run forward pass to trigger the hook
    outputs = target_model(**inputs)
    hidden_states = hidden_states[0]
    
    # Clean up
    hook_handle.remove()
    return hidden_states

### Extract Hidden States from Target Model

We'll extract hidden states from layer 15 of Llama-3.1-8B using text about Machine Learning from Wikipedia. These hidden states capture the model's internal representation of the input text at that specific layer.

In [4]:
layer = 15

# Load demo text: 4 lines about Machine Learning from Wikipedia
# This provides concrete examples for feature extraction and description
ml_demo_texts = open("./demo_data/MachineLearning_wikipedia.txt", "r").read().split("\n")

print(f"Extracting hidden states from layer {layer} of {target_model_name}")
print(f"Processing {len(ml_demo_texts)} text samples...")

# Extract the internal representations that we'll ask models to describe
extracted_hidden_states = extract_from_residual_stream(target_model_path, layer, ml_demo_texts)

print(f"Successfully extracted {len(extracted_hidden_states)} hidden state vectors")
print(f"Each hidden state has dimension: {extracted_hidden_states[0].shape}")

`torch_dtype` is deprecated! Use `dtype` instead!


Extracting hidden states from layer 15 of Llama-3.1-8B
Processing 4 text samples...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Successfully extracted 4 hidden state vectors
Each hidden state has dimension: torch.Size([4096])


### Description Generation Function

This function takes an explainer model and generates natural language descriptions of the extracted features using the continuous feature placeholders.

In [5]:
import torch

@torch.no_grad()
def generate_description(model, tokenizer, input_example, target_feature):
    """
    Generate a natural language description of a continuous feature.
    
    Args:
        model: The explainer model (with continuous feature support)
        tokenizer: Tokenizer for the explainer model
        input_example: Template string with feature placeholder
        target_feature: The actual feature vector to describe
        
    Returns:
        String description of the feature
    """
    model.eval()
    
    # Configure generation stopping criteria
    if "ContinuousQwen3ForCausalLM" in model.config.architectures:
        stop_strings = None
    else:
        stop_strings = [" [END]", "\n", ">>>."]

    # Tokenize the input template
    tokenized_inputs = tokenizer(
        [input_example],
        padding=False,
        truncation=True,
        return_tensors="pt",
        padding_side="left",
    )

    # Generate description using the continuous feature
    generation_outputs = model.generate(
        input_ids=tokenized_inputs["input_ids"].to(model.device),
        attention_mask=tokenized_inputs["attention_mask"].to(model.device),
        max_new_tokens=30,  # Limit description length
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        do_sample=False,  # Use greedy decoding for consistent results
        temperature=None,
        top_p=None,
        inputs_continuous_tokens=[target_feature],  # Pass the feature vector
        return_dict_in_generate=True,
        output_hidden_states=True,
        stop_strings=stop_strings,
        tokenizer=tokenizer,
        extra_args=[],
    )

    # Extract the generated description
    prediction = tokenizer.decode(
        generation_outputs["sequences"][0][len(tokenized_inputs["input_ids"][0]) :],
        skip_special_tokens=True,
    ).strip()
    return prediction

### Experiment 1: Generate Descriptions for Residual Stream Features

Now we'll test the privileged access hypothesis by comparing how well each explainer model can describe the hidden states extracted from the target model. We expect the self-explainer (Llama-3.1-8B describing itself) to produce the most accurate descriptions.

In [6]:
# Initialize results storage for each text sample
results = {line_id: {"input_text": line_text} for line_id, line_text in enumerate(ml_demo_texts)}

# Scaling factor for feature vectors (may need adjustment for different models)
feature_scale = 1

print("Testing Privileged Access Hypothesis - Task 1: Residual Stream Features")
print("=" * 70)

# Test each explainer model
for explainer_model_name, explainer_model_path in feature_explainer_models.items():
    print(f"\nUsing {explainer_model_name} to explain {target_model_name} features...")

    # Load tokenizers for both explainer and target models
    model_tokenizer = AutoTokenizer.from_pretrained(config["model_path"], padding_side="left")
    model_tokenizer.pad_token = model_tokenizer.eos_token
    model_tokenizer.pad_token_id = model_tokenizer.eos_token_id

    target_tokenizer = AutoTokenizer.from_pretrained(target_model_path, padding_side="left")
    target_tokenizer.pad_token = target_tokenizer.eos_token
    target_tokenizer.pad_token_id = target_tokenizer.eos_token_id

    # Set up special tokens for continuous features
    special_token_ids = {
        cont_token: model_tokenizer.convert_tokens_to_ids(config["continuous_tokens"][cont_token])
        for cont_token in config["continuous_tokens"]
    }

    # Load the explainer model
    model, target_model, _ = load_models(
        predictor_model_path=explainer_model_path,
        target_model_path=target_model_path,
        special_tokens_ids=special_token_ids,
        cache_dir=None,
        train_self=False,
        use_bf16=config["train"].get("bf16", True),
        batch_size=config["test"]["batch_size"],
        use_embed_proj=config.get("use_embed_proj", False),
        embed_proj_path=config.get("embed_proj_path", None),
        predictor_model_type=config["model_path"],
    )

    # Generate descriptions for each hidden state
    for i in range(len(ml_demo_texts)):
        prediction = generate_description(
            model,
            model_tokenizer,
            input_with_placeholder[i % len(input_with_placeholder)],  # Cycle through templates
            extracted_hidden_states[i] * feature_scale,
        )
        results[i][explainer_model_name] = prediction

    # Clean up to save GPU memory
    del model, target_model
    torch.cuda.empty_cache()
    print(f"Completed descriptions for {explainer_model_name}")

# Display results for comparison
print("\n" + "=" * 70)
print(f"RESULTS - Residual Stream Feature Descriptions (Layer {layer})")
print("=" * 70)

for line_id, result in results.items():
    print(f"\nInput Text {line_id + 1}:")
    print(f"   {result['input_text']}")
    print(f"\nHidden State Descriptions:")
    
    for explainer_model_name in feature_explainer_models.keys():
        privilege_indicator = "[SELF]" if explainer_model_name == target_model_name else "[EXTERNAL]"
        print(f"   {privilege_indicator} {explainer_model_name}:")
        print(f"      {result[explainer_model_name]}")
    print("-" * 50)

Testing Privileged Access Hypothesis - Task 1: Residual Stream Features

Using Llama-3.1-8B to explain Llama-3.1-8B features...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading model of type:  llama from:  Transluce/features_explain_llama3.1_8b_llama3.1_8b


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Completed descriptions for Llama-3.1-8B-Instruction

Using Llama-3-8B to explain Llama-3.1-8B features...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading model of type:  llama from:  Transluce/features_explain_llama3.1_8b_llama3_8b


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Completed descriptions for Llama-3-8B

RESULTS - Residual Stream Feature Descriptions (Layer 15)

Input Text 1:
   Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions. Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.

Hidden State Descriptions:
   [SELF] Llama-3.1-8B:
      technical terms and symbols related to programming or mathematical equations in a document context.
   [EXTERNAL] Llama-3.1-8B-Instruction:
      8 or 9 in various contexts data structures or algorithms performance evaluations analyses research papers and technical content academic discussions cybersecurity and
   [EXTERNAL] Llama-3-8B:
      references to speci

## 3. Task 2: Sparse Autoencoder (SAE) Feature Description

### Experiment 2: Generate Descriptions for SAE Features

In this second experiment, we test the privileged access hypothesis using Sparse Autoencoder (SAE) features. SAEs decompose model activations into interpretable features, and we have ground truth descriptions from Neuronpedia for comparison.

In [7]:
import json
import pickle

print("Testing Privileged Access Hypothesis - Task 2: SAE Features")
print("=" * 70)

# Load pre-computed SAE features and ground truth explanations
layer = 15
print(f"Loading cached SAE features and explanations for layer {layer}...")

# Note: Full dataset can be downloaded with:
# wget https://transluce-public.s3.us-east-1.amazonaws.com/introspective-interp/SAE_feature_explanations_llama3.1_8b.tar.gz
# tar -xzvf SAE_feature_explanations_llama3.1_8b.tar.gz

demo_explanations = json.load(open(f"./demo_data/three_{layer}layer_sae_feature_descriptions.json", "r"))
demo_features = pickle.load(open(f"./demo_data/three_{layer}layer_sae_features.pkl", "rb"))
demo_feature_ids = [dee[0][1] for dee in demo_explanations]

print(f"Loaded {len(demo_feature_ids)} SAE features with ground truth labels")

# Initialize results with ground truth labels from Neuronpedia
results = {
    feature_id: {"neuronpedia_label": demo_explanations[i][1]["chosen_label"]} 
    for i, feature_id in enumerate(demo_feature_ids)
}

# Scaling factor for SAE features (typically need higher scaling than residual stream)
feature_scale = 10.0

# Test each explainer model
for explainer_model_name, explainer_model_path in feature_explainer_models.items():
    print(f"\nUsing {explainer_model_name} to explain SAE features...")

    # Load tokenizers
    model_tokenizer = AutoTokenizer.from_pretrained(config["model_path"], padding_side="left")
    model_tokenizer.pad_token = model_tokenizer.eos_token
    model_tokenizer.pad_token_id = model_tokenizer.eos_token_id

    target_tokenizer = AutoTokenizer.from_pretrained(target_model_path, padding_side="left")
    target_tokenizer.pad_token = target_tokenizer.eos_token
    target_tokenizer.pad_token_id = target_tokenizer.eos_token_id

    # Configure special tokens for continuous features
    special_token_ids = {
        cont_token: model_tokenizer.convert_tokens_to_ids(config["continuous_tokens"][cont_token])
        for cont_token in config["continuous_tokens"]
    }

    # Load the explainer model
    model, target_model, _ = load_models(
        predictor_model_path=explainer_model_path,
        target_model_path=target_model_path,
        special_tokens_ids=special_token_ids,
        cache_dir=None,
        train_self=False,
        use_bf16=config["train"].get("bf16", True),
        batch_size=config["test"]["batch_size"],
        use_embed_proj=config.get("use_embed_proj", False),
        embed_proj_path=config.get("embed_proj_path", None),
        predictor_model_type=config["model_path"],
    )

    # Generate descriptions for each SAE feature
    for i in range(len(demo_features)):
        prediction = generate_description(
            model,
            model_tokenizer,
            input_with_placeholder[i],
            demo_features[i] * feature_scale,
        )
        results[demo_feature_ids[i]][explainer_model_name] = prediction

    # Clean up GPU memory
    del model, target_model
    torch.cuda.empty_cache()
    print(f"✓ Completed descriptions for {explainer_model_name}")

# Display results comparing to ground truth
print("\n" + "=" * 70)
print(f"RESULTS - SAE Feature Descriptions (Layer {layer})")
print("=" * 70)

for feature_id, result in results.items():
    print(f"\nFeature ID: {feature_id}")
    print(f"Neuronpedia Description: {result['neuronpedia_label']}")
    print(f"\nModel Descriptions:")

    for explainer_model_name in feature_explainer_models.keys():
        privilege_indicator = "[SELF]" if explainer_model_name == target_model_name else "[EXTERNAL]"
        print(f"   {privilege_indicator} {explainer_model_name}:")
        print(f"      {result[explainer_model_name]}")
    print("-" * 50)

Testing Privileged Access Hypothesis - Task 2: SAE Features
Loading cached SAE features and explanations for layer 15...
Loaded 3 SAE features with ground truth labels

Using Llama-3.1-8B to explain SAE features...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading model of type:  llama from:  Transluce/features_explain_llama3.1_8b_llama3.1_8b


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Completed descriptions for Llama-3.1-8B

Using Llama-3.1-8B-Instruction to explain SAE features...


...

✓ Completed descriptions for Llama-3.1-8B-Instruction

Using Llama-3-8B to explain SAE features...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading model of type:  llama from:  Transluce/features_explain_llama3.1_8b_llama3_8b


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Completed descriptions for Llama-3-8B

RESULTS - SAE Feature Descriptions (Layer 15)

Feature ID: 24243
Neuronpedia Description:  terms related to database queries and data retrieval processes

Model Descriptions:
   [SELF] Llama-3.1-8B:
      references to specific individuals or entities in a context related to events or actions in a narrative format, particularly in sports or competitive scenarios.
   [EXTERNAL] Llama-3.1-8B-Instruction:
      8-bit data file types and attributes metadata data structures operations related to file management and data manipulation programming concepts algorithms security
   [EXTERNAL] Llama-3-8B:
      references to specific locations or entities, particularly in a formal or structured context such as addresses or organizational names related to the United States Postal Service and similar organizations
--------------------------------------------------

Feature ID: 56673
Neuronpedia Description: references to sports rankings and achievements

Mode

## References

1. Li, B. Z., Guo, Z. C., Huang, V., Steinhardt, J., & Andreas, J. (2025). *Training Language Models to Explain Their Own Computations*. arXiv:2511.08579.

2. Neuronpedia: [https://neuronpedia.org/](https://neuronpedia.org/) - Community-driven interpretability platform with SAE feature descriptions
